# Rank a compound library from measured activity

This is a one-round active-learning starting point: train on known compounds, check performance on held-out structures, then rank an unmeasured library. Replace the example CSV with your own data in the next cell. Lower values can be configured as better (for example, dG).

From the repository root, install `requirements-notebook.txt` and open this notebook in JupyterLab. A CPU is enough for the small example.

In [ ]:
# Configuration — edit this cell for your experiment.
MEASURED_CSV = 'data/TYK2.csv'
SMILES_COLUMN = 'SMILES'
ACTIVITY_COLUMN = 'affinity'
LOWER_IS_BETTER = False
LIBRARY_CSV = None  # Example: 'my_library.csv'; None uses held-out measured rows as a demo.
LIBRARY_SMILES_COLUMN = 'SMILES'
TOP_N = 20
OUTPUT_DIR = 'notebook_outputs'
RANDOM_SEED = 42

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import rdFingerprintGenerator
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import GroupShuffleSplit

RDLogger.DisableLog('rdApp.warning')
repo = Path.cwd()
if not (repo / 'data' / 'TYK2.csv').exists():
    repo = repo.parent
if not (repo / 'data' / 'TYK2.csv').exists():
    raise FileNotFoundError('Run Jupyter from the repository root or notebooks/ directory.')

def input_path(name):
    path = Path(name)
    return path if path.is_absolute() else repo / path

def prepare(csv_path, smiles_column, activity_column=None):
    frame = pd.read_csv(input_path(csv_path))
    needed = [smiles_column] + ([activity_column] if activity_column else [])
    missing = set(needed) - set(frame.columns)
    if missing:
        raise ValueError(f'{csv_path} is missing columns: {sorted(missing)}')
    frame = frame[needed].dropna().copy()
    molecules = [Chem.MolFromSmiles(str(s)) for s in frame[smiles_column]]
    valid = [m is not None for m in molecules]
    if not all(valid):
        print(f'{csv_path}: skipped {len(valid) - sum(valid)} invalid SMILES')
    frame = frame.loc[valid].copy()
    molecules = [m for m in molecules if m is not None]
    frame['SMILES'] = [Chem.MolToSmiles(m, isomericSmiles=True) for m in molecules]
    frame['connectivity'] = [Chem.MolToSmiles(m, isomericSmiles=False) for m in molecules]
    if activity_column:
        frame['activity'] = pd.to_numeric(frame[activity_column], errors='coerce')
        frame = frame.dropna(subset=['activity'])
    frame = frame.drop_duplicates('SMILES').reset_index(drop=True)
    if frame.empty:
        raise ValueError(f'{csv_path} has no usable rows')
    return frame

fingerprinter = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048, includeChirality=True)
def fingerprints(smiles):
    array = np.zeros((len(smiles), 2048), dtype=np.uint8)
    for i, smi in enumerate(smiles):
        DataStructs.ConvertToNumpyArray(fingerprinter.GetFingerprint(Chem.MolFromSmiles(smi)), array[i])
    return array

measured = prepare(MEASURED_CSV, SMILES_COLUMN, ACTIVITY_COLUMN)
if len(measured) < 20 or measured.connectivity.nunique() < 5:
    raise ValueError('Provide at least 20 measured molecules from at least 5 connectivity groups.')
print(f'{len(measured)} measured molecules; {measured.connectivity.nunique()} connectivity groups')

## Check generalization

Hold out whole connectivity groups. These predictions are for evaluation only; the final library model is refit below using all measured rows.

In [ ]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_SEED)
train_idx, test_idx = next(splitter.split(measured, groups=measured.connectivity))
assert not set(measured.connectivity.iloc[train_idx]) & set(measured.connectivity.iloc[test_idx])
X = fingerprints(measured.SMILES)
y = measured.activity.to_numpy()
model = RandomForestRegressor(n_estimators=200, min_samples_leaf=2, max_features=0.5, n_jobs=-1, random_state=RANDOM_SEED)
model.fit(X[train_idx], y[train_idx])
pred = model.predict(X[test_idx])
validation = pd.DataFrame({'SMILES': measured.SMILES.iloc[test_idx].to_numpy(), 'true_activity': y[test_idx], 'predicted_activity': pred})
print(f'Holdout: {len(test_idx)} molecules | MAE {mean_absolute_error(y[test_idx], pred):.3f} | R² {r2_score(y[test_idx], pred):.3f} | Spearman {validation.true_activity.corr(validation.predicted_activity, method="spearman"):.3f}')
validation.head()

## Rank the library

With `LIBRARY_CSV = None`, the held-out molecules are the demonstration library. Set a library CSV to train on every measured molecule and rank new structures. Molecules already present in the measured file are removed.

In [ ]:
if LIBRARY_CSV is None:
    library = measured.iloc[test_idx][['SMILES']].copy().reset_index(drop=True)
    print('Demo mode: ranking the held-out molecules.')
else:
    library = prepare(LIBRARY_CSV, LIBRARY_SMILES_COLUMN)[['SMILES']].copy()
    library = library.loc[~library.SMILES.isin(measured.SMILES)].reset_index(drop=True)
    if library.empty:
        raise ValueError('No unmeasured library molecules remain after overlap removal.')
    model.fit(X, y)
    print(f'Ranking {len(library)} unmeasured molecules with a model fit to all {len(measured)} measured molecules.')
library['predicted_activity'] = model.predict(fingerprints(library.SMILES))
library = library.sort_values('predicted_activity', ascending=LOWER_IS_BETTER).reset_index(drop=True)
library.insert(0, 'rank', np.arange(1, len(library) + 1))
top = library.head(min(TOP_N, len(library))).copy()
out = input_path(OUTPUT_DIR)
out.mkdir(parents=True, exist_ok=True)
validation.to_csv(out / 'validation_predictions.csv', index=False)
library.to_csv(out / 'full_ranked_library.csv', index=False)
top.to_csv(out / 'top_candidates.csv', index=False)
print(f'Saved {len(library)} library ranks and {len(top)} top candidates to {out}')
top.head(10)

### Next round

Measure a batch from `top_candidates.csv`, append their observed activities to your measured CSV, and rerun the notebook. Keep a record of the assay conditions and avoid treating the model's predictions as measured values.